# CUDA port on a Colab GPU

Builds the C++ CPU backend and the CUDA backend, runs the Orszag-Tang vortex
on the GPU, checks it field by field against the NumPy reference state in the
repository, then times all three implementations on the same machine.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
!nvcc --version | tail -2

In [ ]:
import os
REPO = "https://github.com/MehrasAmirhakimi/GPU-MHD-Orszag-Tang-2D.git"
if not os.path.isdir("GPU-MHD-Orszag-Tang-2D"):
    !git clone -q {REPO}
%cd GPU-MHD-Orszag-Tang-2D
!git pull -q
!git log --oneline -1

## Build

`ARCH=native` lets nvcc detect the card. If that fails on an older nvcc, name
it instead: `sm_75` for a T4, `sm_80` for an A100, `sm_89` for an L4.

In [ ]:
!make -C cuda cpu gpu ARCH=native

## Validation

Same grid, same diffusivities and the same end time as the stored reference
`ot_state.npz` (256², t = 0.5). `compare.py` exits with PASS if every field
agrees with NumPy to 1e-10 relative to that field's size; a correct port lands
near 1e-13.

If it fails, rebuild with `make -C cuda gpu DEBUG_SYNC=1` and run again: that
synchronises after every kernel, so an error is reported at the kernel that
caused it.

In [ ]:
!cd cuda && ./mhd2d_gpu --nx 256 --tmax 0.5 --every 40
!python compare.py ot_state.npz cuda/ot_state_gpu.npy

## Timing

A fixed number of steps per grid size, so every run does the same work per
step. The NumPy figure comes from `run_ot.py` over a short run, and the C++
CPU backend uses every core Colab provides. All three are float64.

This is the naive port: global memory only, no tiling, no kernel fusion. It
is the baseline the optimisation stage has to beat.

In [ ]:
import os, re, subprocess

def per_point_us(cmd, cwd="."):
    out = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return float(re.search(r"\(([0-9.eE+-]+) us per point-step", out.stdout).group(1))

rows = []

# NumPy reference, single core. Run in its own folder: run_ot.py writes
# ot_state.npz, and it must not overwrite the reference state in the repo root.
os.makedirs("timing_numpy", exist_ok=True)
us = per_point_us("python ../run_ot.py --nx 256 --tmax 0.04 --no-plot", cwd="timing_numpy")
rows.append(("NumPy", 256, us))

# C++ CPU backend, OpenMP
for n, steps in [(256, 50), (512, 20)]:
    us = per_point_us(f"./mhd2d_cpu --nx {n} --steps {steps} --every 0 --quiet "
                      f"--out /dev/null --csv /dev/null", cwd="cuda")
    rows.append(("C++ CPU", n, us))

# CUDA backend
for n, steps in [(256, 200), (512, 100), (1024, 50), (2048, 20)]:
    us = per_point_us(f"./mhd2d_gpu --nx {n} --steps {steps} --every 0 --quiet "
                      f"--out /dev/null --csv /dev/null", cwd="cuda")
    rows.append(("CUDA", n, us))

numpy_us = rows[0][2]
print("| backend | grid | us per point-step | Mpoint-steps/s | speed-up over NumPy |")
print("| --- | --- | --- | --- | --- |")
for name, n, us in rows:
    print(f"| {name} | {n}^2 | {us:.4g} | {1/us:.1f} | {numpy_us/us:.0f}x |")

Copy the table above into the README, together with the GPU name from the
first cell.